# 🤖 Phase 5 — Demand Forecasting (Machine Learning)
**Project**: Blinkit Supply Chain & Inventory Optimization  
**Author**: Rishi Verma

---
### 🎯 Goals:
- Build ML models to predict `quantity` sold per product
- Models compared: Linear Regression, Random Forest, XGBoost
- Evaluate with RMSE, MAE, R² score
- Feature importance analysis
- Forecast next 30 days demand per category
- Export predictions for Power BI
---

## 1️⃣ Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('⚠️  XGBoost not installed — using GradientBoosting as substitute')

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

CLEANED = '../data/cleaned'
CHARTS  = '../outputs/charts'
REPORTS = '../outputs/reports'
os.makedirs(REPORTS, exist_ok=True)

# Load data
master_sales = pd.read_csv(f'{CLEANED}/master_sales.csv', parse_dates=['order_date'])
products     = pd.read_csv(f'{CLEANED}/products_cleaned.csv')
inventory    = pd.read_csv(f'{CLEANED}/inventory_cleaned.csv', parse_dates=['date'])

print('✅ Libraries loaded & data ready!')
print(f'master_sales: {master_sales.shape}')

## 2️⃣ Feature Engineering for ML

In [ ]:
df = master_sales.copy()

# ── Time-based features ──
df['day_of_week']  = df['order_date'].dt.dayofweek        # 0=Mon, 6=Sun
df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
df['week_of_year'] = df['order_date'].dt.isocalendar().week.astype(int)
df['quarter']      = df['order_date'].dt.quarter

# ── Price features ──
df['price_to_mrp_ratio'] = (df['price'] / df['mrp'].replace(0, np.nan)).round(4)
df['discount_pct']       = df.get('discount_pct', ((df['mrp'] - df['price']) / df['mrp'].replace(0,np.nan) * 100).round(2))

# ── Stock features ──
df['stock_range']   = df['max_stock_level'] - df['min_stock_level']
df['is_perishable'] = (df['shelf_life_days'] <= 7).astype(int)

# ── Encode categoricals ──
le = LabelEncoder()
for col in ['category', 'customer_segment', 'delivery_status']:
    if col in df.columns:
        df[f'{col}_enc'] = le.fit_transform(df[col].fillna('Unknown'))

# ── Select features ──
FEATURE_COLS = [
    'unit_price', 'price', 'mrp', 'margin_percentage', 'shelf_life_days',
    'min_stock_level', 'max_stock_level', 'stock_range', 'is_perishable',
    'price_to_mrp_ratio', 'discount_pct',
    'order_month', 'order_hour', 'day_of_week', 'is_weekend',
    'week_of_year', 'quarter',
    'category_enc'
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
TARGET = 'quantity'

# Drop rows with any nulls in features or target
model_df = df[FEATURE_COLS + [TARGET]].dropna()

print(f'✅ Feature engineering done!')
print(f'   Model dataset shape : {model_df.shape}')
print(f'   Features used       : {len(FEATURE_COLS)}')
print(f'   Target              : {TARGET}')
print(f'\n   Feature list:\n   {FEATURE_COLS}')

## 3️⃣ Train / Test Split

In [ ]:
X = model_df[FEATURE_COLS]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train set : {X_train.shape[0]:,} samples')
print(f'Test set  : {X_test.shape[0]:,} samples')
print(f'\nTarget distribution:')
print(f'  Mean  : {y.mean():.2f}')
print(f'  Median: {y.median():.2f}')
print(f'  Std   : {y.std():.2f}')
print(f'  Min   : {y.min()}')
print(f'  Max   : {y.max()}')

## 4️⃣ Train & Evaluate All Models

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    """Train, predict, and return metrics dict."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred = np.clip(y_pred, 0, None)  # no negative predictions

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    # Cross-validation R2
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')

    return {
        'Model'   : name,
        'RMSE'    : round(rmse, 4),
        'MAE'     : round(mae, 4),
        'R2_Score': round(r2, 4),
        'CV_R2_Mean': round(cv_scores.mean(), 4),
        'CV_R2_Std' : round(cv_scores.std(), 4),
        'predictions': y_pred,
        'model_obj'  : model
    }

# Define models
models = {
    'Linear Regression' : LinearRegression(),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost' if XGBOOST_AVAILABLE else 'Gradient Boosting':
        XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1,
                     random_state=42, verbosity=0) if XGBOOST_AVAILABLE
        else GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
}

print('🤖 Training models...\n')
results = []
for name, model in models.items():
    print(f'  ⏳ Training {name}...')
    res = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    results.append(res)
    print(f'     RMSE={res["RMSE"]:.4f} | MAE={res["MAE"]:.4f} | R²={res["R2_Score"]:.4f} | CV_R²={res["CV_R2_Mean"]:.4f} ± {res["CV_R2_Std"]:.4f}')

print('\n✅ All models trained!')

In [ ]:
# ── Model Comparison Table ──
metrics_df = pd.DataFrame([{k: v for k, v in r.items()
                             if k not in ('predictions', 'model_obj')} for r in results])
print('\n📊 MODEL COMPARISON')
print('='*70)
print(metrics_df.to_string(index=False))
print('='*70)

best_model_name = metrics_df.loc[metrics_df['R2_Score'].idxmax(), 'Model']
best_r2         = metrics_df['R2_Score'].max()
print(f'\n🏆 Best Model: {best_model_name} (R² = {best_r2:.4f})')

In [ ]:
# ── Model Comparison Bar Chart ──
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
model_names = metrics_df['Model'].tolist()
colors = ['#3498db', '#2ecc71', '#e74c3c']

for ax, metric, label, higher_better in zip(
    axes,
    ['R2_Score', 'RMSE', 'MAE'],
    ['R² Score (higher = better)', 'RMSE (lower = better)', 'MAE (lower = better)'],
    [True, False, False]
):
    vals = metrics_df[metric].tolist()
    bar_colors = colors.copy()
    best_idx = vals.index(max(vals) if higher_better else min(vals))
    bar_colors[best_idx] = '#f39c12'   # highlight best
    bars = ax.bar(model_names, vals, color=bar_colors, alpha=0.85)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('🤖 ML Model Comparison — Demand Forecasting',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{CHARTS}/16_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 16_model_comparison.png')

## 5️⃣ Feature Importance (Best Model)

In [ ]:
# Get best model object
best_result = max(results, key=lambda x: x['R2_Score'])
best_model  = best_result['model_obj']

if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=FEATURE_COLS)
    importances = importances.sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(14, 7))
    colors_fi = ['#e74c3c' if i < 5 else '#3498db' for i in range(len(importances))]
    bars = ax.barh(importances.index[::-1], importances.values[::-1], color=colors_fi[::-1])
    ax.set_title(f'🔍 Feature Importance — {best_result["Model"]}',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score')
    for bar, val in zip(bars, importances.values[::-1]):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{CHARTS}/17_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved: 17_feature_importance.png')
    print('\n🔝 Top 5 Most Important Features:')
    for feat, imp in importances.head(5).items():
        print(f'   {feat:<30} : {imp:.4f}')
else:
    # Linear Regression — show coefficients
    coefs = pd.Series(np.abs(best_model.coef_), index=FEATURE_COLS).sort_values(ascending=False)
    print('Linear Regression — Top coefficients (absolute):')
    print(coefs.head(10))

## 6️⃣ Actual vs Predicted Plot

In [ ]:
best_preds = best_result['predictions']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: Actual vs Predicted
axes[0].scatter(y_test, best_preds, alpha=0.4, color='#3498db', s=20)
min_val = min(y_test.min(), best_preds.min())
max_val = max(y_test.max(), best_preds.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Quantity')
axes[0].set_ylabel('Predicted Quantity')
axes[0].set_title(f'📍 Actual vs Predicted — {best_result["Model"]}\n(R² = {best_result["R2_Score"]:.4f})',
                  fontweight='bold')
axes[0].legend()

# Residuals Distribution
residuals = y_test.values - best_preds
axes[1].hist(residuals, bins=30, color='#2ecc71', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[1].set_title('📊 Residuals Distribution', fontweight='bold')
axes[1].set_xlabel('Residual (Actual − Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.suptitle('🤖 Model Prediction Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{CHARTS}/18_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 18_actual_vs_predicted.png')

## 7️⃣ 30-Day Demand Forecast by Category

In [ ]:
# ── Category-level rolling demand + simple next-30-day forecast ──
category_daily = (
    master_sales
    .merge(master_sales[['order_date']].assign(
        order_date=master_sales['order_date']), how='left')
    .groupby(['category', 'order_date'])['quantity']
    .sum()
    .reset_index()
)

# 7-day rolling average per category as the forecast baseline
category_daily = category_daily.sort_values(['category','order_date'])
category_daily['rolling_7d'] = (
    category_daily.groupby('category')['quantity']
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

# Forecast: use last 7-day avg per category as next-30-day daily forecast
last_date = master_sales['order_date'].max()
forecast_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=30, freq='D')

category_avg = (
    category_daily.groupby('category')['rolling_7d']
    .last()
    .reset_index()
    .rename(columns={'rolling_7d': 'daily_forecast'})
)

# Build forecast dataframe
forecast_rows = []
for _, row in category_avg.iterrows():
    for d in forecast_dates:
        forecast_rows.append({
            'category'       : row['category'],
            'forecast_date'  : d,
            'daily_forecast' : round(row['daily_forecast'], 1),
            'forecast_type'  : 'Rolling 7-Day Avg'
        })
forecast_df = pd.DataFrame(forecast_rows)

# 30-day total forecast per category
forecast_summary = forecast_df.groupby('category')['daily_forecast'].sum().reset_index()
forecast_summary.columns = ['category', 'forecast_30d_units']
forecast_summary = forecast_summary.sort_values('forecast_30d_units', ascending=False)

print('📅 30-Day Demand Forecast by Category:')
print('='*45)
for _, r in forecast_summary.iterrows():
    print(f'  {r.category:<25} : {r.forecast_30d_units:>8.0f} units')
print('='*45)
print(f'  TOTAL FORECAST (all categories): {forecast_summary.forecast_30d_units.sum():,.0f} units')

In [ ]:
# ── Forecast Chart ──
top_cats = forecast_summary.head(5)['category'].tolist()
plot_data = category_daily[category_daily['category'].isin(top_cats)]
plot_forecast = forecast_df[forecast_df['category'].isin(top_cats)]

fig, ax = plt.subplots(figsize=(16, 7))
colors_line = sns.color_palette('tab10', 5)

for i, cat in enumerate(top_cats):
    hist = plot_data[plot_data['category'] == cat]
    fcast = plot_forecast[plot_forecast['category'] == cat]
    ax.plot(hist['order_date'], hist['rolling_7d'],
            color=colors_line[i], linewidth=2, label=cat)
    ax.plot(fcast['forecast_date'], fcast['daily_forecast'],
            color=colors_line[i], linewidth=2.5, linestyle='--', alpha=0.8)

ax.axvline(last_date, color='black', linestyle=':', linewidth=2, label='Forecast Start')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 100],
                  last_date, forecast_dates[-1], alpha=0.05, color='grey')
ax.set_title('📅 30-Day Demand Forecast — Top 5 Categories\n(Solid=Historical, Dashed=Forecast)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Daily Units Sold (7-day rolling avg)')
ax.set_xlabel('Date')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{CHARTS}/19_demand_forecast_30day.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 19_demand_forecast_30day.png')

## 8️⃣ Export Predictions & Forecast

In [ ]:
# Export test predictions
test_results = X_test.copy()
test_results['actual_quantity']    = y_test.values
test_results['predicted_quantity'] = best_preds
test_results['error']              = test_results['actual_quantity'] - test_results['predicted_quantity']
test_results['abs_error']          = test_results['error'].abs()

test_results.to_csv(f'{REPORTS}/ml_predictions.csv', index=False)
forecast_df.to_csv(f'{REPORTS}/demand_forecast_30days.csv', index=False)
forecast_summary.to_csv(f'{REPORTS}/forecast_summary_by_category.csv', index=False)
metrics_df.drop(columns=['predictions','model_obj'], errors='ignore').to_csv(
    f'{REPORTS}/model_comparison_metrics.csv', index=False)

print('✅ Exported:')
print(f'   📄 outputs/reports/ml_predictions.csv')
print(f'   📄 outputs/reports/demand_forecast_30days.csv')
print(f'   📄 outputs/reports/forecast_summary_by_category.csv')
print(f'   📄 outputs/reports/model_comparison_metrics.csv')
print()
print('='*60)
print('       🤖 PHASE 5 — ML FORECASTING SUMMARY')
print('='*60)
print(f'  Models trained      : {len(results)}')
print(f'  Features used       : {len(FEATURE_COLS)}')
print(f'  Training samples    : {X_train.shape[0]:,}')
print(f'  Test samples        : {X_test.shape[0]:,}')
print(f'  🏆 Best model       : {best_result["Model"]}')
print(f'  Best R² Score       : {best_result["R2_Score"]:.4f}')
print(f'  Best RMSE           : {best_result["RMSE"]:.4f}')
print(f'  Best MAE            : {best_result["MAE"]:.4f}')
print(f'  30-Day Forecast     : {forecast_summary.forecast_30d_units.sum():,.0f} total units')
print('='*60)
print('  ✅ Phase 5 Complete! Ready for Phase 6 — Power BI')
print('='*60)